In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA gold;

--Create Winrate_TopLane Patch 26.11
CREATE OR REPLACE TABLE Winrate_TopLane_Patch_26_11 AS
SELECT
    p.champion,
    COUNT(*) AS games,
    ROUND(SUM(INT(p.win))/COUNT(*), 2) as winrate,
    ROUND(SUM(CASE WHEN g.game_duration_minutes <= 25 THEN INT(p.win) END)/COUNT(CASE WHEN g.game_duration_minutes <= 25 THEN 1 END), 2) AS early_game_wr,
    ROUND(SUM(CASE WHEN g.game_duration_minutes > 25 THEN INT(p.win) END)/COUNT(CASE WHEN g.game_duration_minutes > 25 THEN 1 END), 2) AS late_game_wr,
    g.game_version as patch
FROM workspace.silver.player_info as p
INNER JOIN workspace.silver.general_info as g ON p.match_id = g.match_id
WHERE p.lane_position = 'TOP'
AND g.game_version LIKE '16.11%'
GROUP BY p.champion, g.game_version
HAVING COUNT(*) > 20
ORDER BY winrate DESC;

